# Import Libraries

In [24]:
import torch
from torchvision import transforms as T
from utils.set_seed import set_random_seed
from torch.utils.tensorboard import SummaryWriter
from weathernet import WeatherNet
from weathernetplusplus import WeatherNetPlusPlus
from mtl_weathernet import MtlWeatherNet
from weathernet_transformer import WeatherNetTransformer

from data.data import get_dataloaders
import utils.trainer as trainer

device = 'cpu'
if torch.backends.mps.is_available(): device = torch.device('mps')
elif torch.cuda.is_available(): device = torch.device('cuda')
print(device)

cuda


# Select model and checkpoint

In [25]:
# choose model_name from "weathernet", "weathernetpp", "mtl", and "transformer"
# choose checkpoint epoch from [0 - 9]
model_name = "weathernet"

## Load weights
Make sure that the model you selected matches the weights

In [26]:
if model_name == "weathernet":
    checkpoint = 8
    model = WeatherNet()
elif model_name == "weathernetpp":
    checkpoint = 9
    model = WeatherNetPlusPlus()
elif model_name == "mtl":
    checkpoint = 20
    model = MtlWeatherNet()
elif model_name == "transformer":
    checkpoint = 20
    model = WeatherNetTransformer()

saved_state = f"./checkpoints/{model_name}_{checkpoint}.pth"
if saved_state is not None:
    print(f"Loading saved state from {saved_state}")
    model.load_checkpoint(saved_state)
    

Loading saved state from ./checkpoints/weathernet_8.pth


# Set Hyperparameters, Load Dataset

In [27]:
batch_size = 16
set_random_seed(42) # seed for reproducibility

In [28]:
# Load BDD100KPlus dataset
trainloader, valloader, testloader = get_dataloaders(dataset_name="Bdd100kPlus", batch_size=batch_size)

## Get Prediction on Image

In [29]:
def get_prediction_labels(model, outputs):
    if model.get_num_pipelines() == 4: # WeatherNet
        fog_out = outputs[:, 0]
        glare_out = outputs[:, 1]
        weather_out = outputs[:, 2:8]
        tod_out = outputs[:, 8:12]

        # Convert raw logits on binary classifiers to categorical labels
        fog_out = fog_out.squeeze() > 0.5
        glare_out = glare_out.squeeze() > 0.5

        # Convert raw logits on multiclass to categorical labels
        weather_out = torch.argmax(weather_out, dim=1)
        tod_out = torch.argmax(tod_out, dim=1)

        evals = torch.tensor([
            fog_out,
            glare_out,
            weather_out,
            tod_out
        ]).to(device=device)

        return evals
    
    elif model.get_num_pipelines() == 7: # WeatherNet++, MtlWeatherNet, WeatherNetTransformer
        fog_out = outputs[:, 0]
        glare_out = outputs[:, 1]
        road_out = outputs[:, 2:5]
        traffic_out = outputs[:, 5:8]
        weather_out = outputs[:, 8:14]
        scene_out = outputs[:, 14:18]
        tod_out = outputs[:, 18:22]

        # Convert raw logits on binary classifiers to categorical labels
        fog_out = fog_out.squeeze() > 0.5
        glare_out = glare_out.squeeze() > 0.5

        # Convert raw logits on multiclass to categorical labels
        road_out = torch.argmax(road_out, dim=1)
        traffic_out = torch.argmax(traffic_out, dim=1)
        weather_out = torch.argmax(weather_out, dim=1)
        scene_out = torch.argmax(scene_out, dim=1)
        tod_out = torch.argmax(tod_out, dim=1)

        evals = torch.tensor([
            fog_out,
            glare_out,
            road_out,
            traffic_out,
            weather_out,
            scene_out,
            tod_out
        ]).to(device=device)

        return evals

## Convert Tensor Values to English Labels

In [30]:
def tensor_to_english_labels(model, pipeline_labels, pred_labels):
    weather_labels = [
        "Clear",
        "Partly Cloudy",
        "Overcast",
        "Rainy",
        "Snowy",
        "Undefined"
    ]

    tod_labels = [
        "Dawn/Dusk",
        "Daytime",
        "Night",
        "Undefined"
    ]

    fog_labels = [
        "No Fog",
        "Fog"
    ]

    glare_labels = [
        "No Glare",
        "Glare"
    ]

    road_labels = [
        "Dry Road",
        "Wet Road",
        "Snowy Road"
    ]

    traffic_labels = [
        "No Traffic",
        "Low/Moderate Traffic",
        "High Traffic"
    ]

    scene_labels = [
        "Residential",
        "Highway",
        "City Street",
        "Undefined"
    ]

    if model.get_num_pipelines() == 4: # WeatherNet
        # Loop through tensor and create a list of labels
        labels = [
            fog_labels[pipeline_labels[0].item()],
            glare_labels[pipeline_labels[1].item()],
            weather_labels[pipeline_labels[2].item()],
            tod_labels[pipeline_labels[3].item()]
        ]
        pred_labels = [
            fog_labels[pred_labels[0].item()],
            glare_labels[pred_labels[1].item()],
            weather_labels[pred_labels[2].item()],
            tod_labels[pred_labels[3].item()]
        ]
        return labels, pred_labels
    
    elif model.get_num_pipelines() == 7: # WeatherNet++, MtlWeatherNet, WeatherNetTransformer
        # Loop through tensor and create a list of labels
        labels = [
            fog_labels[pipeline_labels[0].item()],
            glare_labels[pipeline_labels[1].item()],
            road_labels[pipeline_labels[2].item()],
            traffic_labels[pipeline_labels[3].item()],
            weather_labels[pipeline_labels[4].item()],
            scene_labels[pipeline_labels[5].item()],
            tod_labels[pipeline_labels[6].item()]
        ]
        pred_labels = [
            fog_labels[pred_labels[0].item()],
            glare_labels[pred_labels[1].item()],
            road_labels[pred_labels[2].item()],
            traffic_labels[pred_labels[3].item()],
            weather_labels[pred_labels[4].item()],
            scene_labels[pred_labels[5].item()],
            tod_labels[pred_labels[6].item()]
        ]
        return labels, pred_labels

## Model Inferencing

In [41]:
import pandas as pd

def model_inferencing(model, dataloader, device, num_images=5, specific_image=None):
    """
    Function to run inference on the model and display the results in a dataframe.
    Args:
        model: The trained model to run inference on.
        dataloader: The dataloader containing the images and labels.
        device: The device to run the model on (cpu or cuda).
        num_images: The number of images to run inference on. (default: 5)
        specific_image: A specific image path to run inference on. (default: None)
    """
    model.eval()
    model.to(device)
    if specific_image is not None:
        num_images = 1
    with torch.no_grad():
        for _ in range(num_images):
            # If specific_image is provided, only run inference on that image
            if specific_image is not None:
                print(f"Image path: {specific_image}")
                img_idx = dataloader.dataset.img_paths.index(specific_image)
                image, pipeline_labels = dataloader.dataset.__getitem__(img_idx)
            else:
                # Get random image and labels
                i = torch.randint(0, len(dataloader.dataset), (1,)).item()
                print(f"Image {i+1}:")

                # Get image and labels
                img_path = dataloader.dataset.img_paths[i]
                print(f"Image path: {img_path}")
                image, pipeline_labels = dataloader.dataset.__getitem__(i)

            # Get predicted tensor
            image = image.to(device)
            pipeline_labels = pipeline_labels.to(device)
            outputs = model(image.unsqueeze(0))
            evals = get_prediction_labels(model, outputs)

            # Convert to english labels
            labels, pred_labels = tensor_to_english_labels(model, pipeline_labels, evals)

            # Dataframe column names
            if model.get_num_pipelines() == 4: # WeatherNet
                column_names = [
                    "Fog",
                    "Glare",
                    "Weather",
                    "Time of Day"
                ]
            elif model.get_num_pipelines() == 7: # WeatherNet++, MtlWeatherNet, WeatherNetTransformer
                column_names = [
                    "Fog",
                    "Glare",
                    "Road",
                    "Traffic",
                    "Weather",
                    "Scene",
                    "Time of Day"
                ]

            # Create dataframe
            df = pd.DataFrame(
                {
                    "Pipeline": column_names,
                    "True Label": labels,
                    "Predicted Label": pred_labels
                }
            )
            display(df)
            


In [43]:
filename = "fdb01e04-f8399a22"
specific_image = f"./data\images/test\{filename}.jpg"
model_inferencing(model, testloader, device, specific_image=None)

Image 7573:
Image path: ./data\images/test\dde91821-3b53eada.jpg


,Pipeline,True Label,Predicted Label
0,Fog,No Fog,No Fog
1,Glare,No Glare,No Glare
2,Weather,Clear,Undefined
3,Time of Day,Daytime,Daytime


Image 9316:
Image path: ./data\images/test\e2a6e716-1bf76c68.jpg


,Pipeline,True Label,Predicted Label
0,Fog,No Fog,No Fog
1,Glare,No Glare,No Glare
2,Weather,Clear,Overcast
3,Time of Day,Daytime,Daytime


Image 8296:
Image path: ./data\images/test\dffc95e3-90e19e41.jpg


,Pipeline,True Label,Predicted Label
0,Fog,No Fog,No Fog
1,Glare,No Glare,No Glare
2,Weather,Clear,Undefined
3,Time of Day,Daytime,Daytime


Image 8568:
Image path: ./data\images/test\e0cca589-073fb17a.jpg


,Pipeline,True Label,Predicted Label
0,Fog,No Fog,No Fog
1,Glare,No Glare,No Glare
2,Weather,Clear,Clear
3,Time of Day,Daytime,Night


Image 4707:
Image path: ./data\images/test\d65cffd8-e5a6fa53.jpg


,Pipeline,True Label,Predicted Label
0,Fog,No Fog,No Fog
1,Glare,No Glare,No Glare
2,Weather,Clear,Clear
3,Time of Day,Daytime,Night
